# Mini SCADA — tkinter vs streamlit

Notebook de révision : pour chaque concept, un rappel **tkinter** puis l'équivalent **streamlit**.

⚠️ Les cellules `streamlit` ci-dessous sont écrites en tant que **fichiers `.py` à exécuter en dehors du notebook** avec la commande :
```bash
streamlit run fichier.py
```
Streamlit ne s'exécute pas directement dans une cellule Jupyter classique — chaque exemple sera donc écrit dans un fichier `.py` via `%%writefile`.

## 1. Créer la fenêtre / l'application

### tkinter (rappel)

In [ ]:
import tkinter as tk

fenetre = tk.Tk()
fenetre.title("Mini SCADA")
fenetre.geometry("300x200")

# ... tout le contenu ici ...

fenetre.mainloop()   # obligatoire : lance la boucle qui garde la fenêtre ouverte

### streamlit (nouveau)

On écrit le code dans un fichier `app_streamlit.py` avec `%%writefile`, puis on le lance depuis un terminal avec `streamlit run app_streamlit.py`.

In [ ]:
%%writefile app_streamlit.py
import streamlit as st

st.title("Mini SCADA")
# ... tout le contenu ici ...
# pas de mainloop() ! le script se relance tout seul a chaque interaction

👉 **Différence clé** : tkinter tourne dans une boucle infinie (`mainloop`) qui attend des événements (clics...). Streamlit n'a pas de boucle : il **ré-exécute tout le fichier de haut en bas** à chaque interaction avec un widget.

## 2. Afficher du texte

### tkinter

In [ ]:
label = tk.Label(fenetre, text="Temperature : 55°C", font=("Arial", 14))
label.pack()   # .pack() = obligatoire pour que le widget s'affiche

### streamlit

In [ ]:
%%writefile -a app_streamlit.py

st.write("Temperature : 55°C")
# ou plus adapte pour une mesure :
st.metric("Temperature", "55 °C")

👉 **Différence clé** : en tkinter, chaque widget doit être « placé » avec `.pack()`, `.grid()` ou `.place()`. En streamlit, on écrit juste `st.xxx(...)` et ça s'affiche automatiquement, dans l'ordre du code.

## 3. Un bouton

### tkinter

In [ ]:
def au_clic():
    print("Bouton clique")

bouton = tk.Button(fenetre, text="Cliquer", command=au_clic)
bouton.pack()

### streamlit

In [ ]:
if st.button("Cliquer"):
    st.write("Bouton clique")

👉 **Différence clé** : en tkinter, on donne une **fonction** au bouton (`command=au_clic`), appelée au clic (logique événementielle classique). En streamlit, `st.button()` retourne `True` juste après le clic — on teste ça avec un `if`.

## 4. Garder un état en mémoire (le point le plus déroutant venant de tkinter)

### tkinter — une variable Python normale suffit

In [ ]:
etat_machine = {"marche": False}

def basculer():
    etat_machine["marche"] = not etat_machine["marche"]
    label_etat.config(text="MARCHE" if etat_machine["marche"] else "ARRET")

bouton = tk.Button(fenetre, text="ON/OFF", command=basculer)
bouton.pack()
label_etat = tk.Label(fenetre, text="ARRET")
label_etat.pack()

### streamlit — il faut `st.session_state`

Une variable normale serait réinitialisée à chaque clic, car **tout le script se relance**.

In [ ]:
if "marche" not in st.session_state:
    st.session_state.marche = False

if st.button("ON/OFF"):
    st.session_state.marche = not st.session_state.marche

st.write("MARCHE" if st.session_state.marche else "ARRET")

👉 **C'est LE piège classique** quand on vient de tkinter : oublier `session_state` et se demander pourquoi la variable « reset » à chaque clic.

## 5. Exemple complet — le même Mini SCADA dans les deux versions

### Version tkinter complète

In [ ]:
import tkinter as tk

fenetre = tk.Tk()
fenetre.title("Mini SCADA")

etat = {"marche": False}

def basculer():
    etat["marche"] = not etat["marche"]
    if etat["marche"]:
        bouton.config(text="MARCHE", bg="green")
    else:
        bouton.config(text="ARRET", bg="red")

tk.Label(fenetre, text="Temperature : 55°C", font=("Arial", 14)).pack(pady=10)
bouton = tk.Button(fenetre, text="ARRET", bg="red", fg="white", command=basculer)
bouton.pack(pady=10)

fenetre.mainloop()

### Version streamlit complète

Cette cellule écrit le fichier final `mini_scada_streamlit.py`. Pour le tester : ouvre un terminal dans le même dossier que ce notebook et lance :
```bash
streamlit run mini_scada_streamlit.py
```

In [ ]:
%%writefile mini_scada_streamlit.py
import streamlit as st

st.title("Mini SCADA")

if "marche" not in st.session_state:
    st.session_state.marche = False

st.metric("Temperature", "55 °C")

if st.button("Basculer ON/OFF"):
    st.session_state.marche = not st.session_state.marche

if st.session_state.marche:
    st.success("Machine en MARCHE")
else:
    st.error("Machine a l'ARRET")

## 🧠 Tableau récap à garder sous la main

| Concept | tkinter | streamlit |
|---|---|---|
| Lancer l'appli | `python fichier.py` | `streamlit run fichier.py` |
| Boucle principale | `fenetre.mainloop()` | pas de boucle, réexécution auto |
| Afficher un widget | `.pack()` / `.grid()` obligatoire | s'affiche tout seul dans l'ordre |
| Réagir à un clic | fonction passée en `command=` | `if st.button(...):` |
| Garder une valeur en mémoire | variable Python classique | `st.session_state` |
| Modifier un widget existant | `.config(text=..., bg=...)` | on ré-affiche avec une nouvelle valeur |

## ✅ Prochaine étape

Une fois ces bases comprises, on pourra enrichir le Mini SCADA avec :
- des colonnes (`st.columns`) pour organiser température / pression / vitesse
- des couleurs de statut dynamiques
- les vraies données issues du CSV `automatisme_donnees.csv`
- éventuellement une actualisation automatique (auto-refresh) pour simuler du temps réel